# Question 3: Association Rule Mining
Dataset: `mobile_price.csv`, filtered to `price_range == 1`  
Features: `ram`, `int_memory`, `px_width`, `battery_power`

In [ ]:
import pandas as pd
import numpy as np
from mlxtend.frequent_patterns import fpgrowth, association_rules

## Data Preparation: Filter, Categorize, Build Transactions

In [ ]:
df = pd.read_csv('mobile_price.csv')

# Filter: only price_range == 1
df_filtered = df[df['price_range'] == 1].copy().reset_index(drop=True)
print(f'Samples with price_range=1: {len(df_filtered)}')

features = ['ram', 'int_memory', 'px_width', 'battery_power']

In [ ]:
def categorize_feature(series):
    """Divide values into low/medium/high using 3:4:3 ratio on value range."""
    min_val = series.min()
    max_val = series.max()
    range_val = max_val - min_val

    low_thresh  = min_val + 0.3 * range_val   # bottom 30%
    high_thresh = min_val + 0.7 * range_val   # top 30% starts here

    print(f'  {series.name:<15} min={min_val}, max={max_val}, '
          f'low<{low_thresh:.1f}, medium<{high_thresh:.1f}, high>={high_thresh:.1f}')

    def label(val):
        if val < low_thresh:
            return 'low'
        elif val < high_thresh:
            return 'medium'
        else:
            return 'high'

    return series.apply(label)

print('Categorization thresholds:')
df_cat = pd.DataFrame()
for feat in features:
    df_cat[feat] = categorize_feature(df_filtered[feat])

print('\nFirst 5 categorized rows:')
print(df_cat.head())

In [ ]:
# Show example transactions
print('Example transactions (first 5):')
for i in range(5):
    transaction = [f'{feat}_{df_cat.loc[i, feat]}' for feat in features]
    print(f'  Row {i}: {transaction}')

In [ ]:
# Build one-hot encoded DataFrame required by mlxtend
# Each column is one item, e.g. 'ram_high', value is True/False
te_df = pd.DataFrame()
for feat in features:
    for level in ['low', 'medium', 'high']:
        item_name = f'{feat}_{level}'
        te_df[item_name] = (df_cat[feat] == level)

print(f'One-hot encoded shape: {te_df.shape}  ({te_df.shape[1]} items, {te_df.shape[0]} transactions)')
te_df.head()

## 3a. Frequent Patterns with support ≥ 0.3 using FP-growth

In [ ]:
freq_itemsets = fpgrowth(te_df, min_support=0.3, use_colnames=True)
freq_itemsets = freq_itemsets.sort_values('support', ascending=False).reset_index(drop=True)

# Add itemset length for clarity
freq_itemsets['length'] = freq_itemsets['itemsets'].apply(len)

print(f'Total frequent itemsets found: {len(freq_itemsets)}')
freq_itemsets

## 3b. Association Rules with support ≥ 0.3, confidence ≥ 0.4, lift ≥ 0.8

In [ ]:
rules = association_rules(freq_itemsets, metric='confidence', min_threshold=0.4)

# Apply remaining filters
rules = rules[
    (rules['support']    >= 0.3) &
    (rules['lift']       >= 0.8)
].reset_index(drop=True)

rules = rules[['antecedents', 'consequents', 'support', 'confidence', 'lift']]
rules = rules.sort_values('lift', ascending=False).reset_index(drop=True)

print(f'Total association rules found: {len(rules)}')
rules.style.format({'support': '{:.4f}', 'confidence': '{:.4f}', 'lift': '{:.4f}'})